<a href="https://colab.research.google.com/github/cybercolombia/suelosabio/blob/feature%2FSCRUM-14/notebooks/ClimatePipeline/07_ClimateMunicipalAudit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ClimateMunicipalAudit

Audita la capa diaria municipal de precipitación sin modificarla ni imputar ausencias.

## Preguntas de esta compuerta

- ¿Qué municipios y periodos tienen cobertura temporal suficiente para construir indicadores?
- ¿Dónde están los días con cobertura insuficiente de estaciones?
- ¿Cuánto cambian los resultados al usar media en lugar de mediana?
- ¿Qué municipios multiestación requieren revisión antes de aprobar la regla v1?

Los umbrales de lluvia son diagnósticos de sensibilidad. Esta auditoría no aprueba un umbral, no elimina filas y no habilita todavía el paso 08.

## 1. Preparar el repositorio

La celda actualiza explícitamente `feature/SCRUM-14` para evitar módulos antiguos en Colab.

In [ ]:
from pathlib import Path

import subprocess
import sys

try:
    from google.colab import drive

    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/cybercolombia/suelosabio.git'
REPO_REF = 'feature/SCRUM-14'
REPO_DIR = Path('/content/suelosabio') if IN_COLAB else Path.cwd()
ACTUALIZAR_REPOSITORIO = True

if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
            check=True,
        )
    elif ACTUALIZAR_REPOSITORIO:
        remote_ref = f'refs/remotes/origin/{REPO_REF}'
        subprocess.run(
            [
                'git', 'fetch', '--depth', '1', 'origin',
                f'+refs/heads/{REPO_REF}:{remote_ref}',
            ],
            cwd=REPO_DIR,
            check=True,
        )
        subprocess.run(
            ['git', 'checkout', '-B', REPO_REF, remote_ref],
            cwd=REPO_DIR,
            check=True,
        )

PIPELINE_DIR = REPO_DIR / 'notebooks' / 'ClimatePipeline'
if not PIPELINE_DIR.exists():
    raise FileNotFoundError(f'No existe la carpeta del pipeline: {PIPELINE_DIR}')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

print({'in_colab': IN_COLAB, 'repo_ref': REPO_REF, 'repo_dir': str(REPO_DIR)})

## 2. Configuración protegida

Primero ejecute todo con `EJECUTAR_AUDITORIA_MUNICIPAL=False` y confirme el plan. Después cambie únicamente esa bandera a `True`.

In [ ]:
import importlib
import json
import time

import pandas as pd

import ClimateProcessingUtils
import PrecipitationMunicipalAudit

importlib.reload(ClimateProcessingUtils)
importlib.reload(PrecipitationMunicipalAudit)

from ClimateProcessingUtils import (
    ahora_proyecto,
    detectar_commit,
    escribir_json_atomico,
    escribir_parquet_atomico,
    escribir_texto_atomico,
    formatear_duracion,
    slugificar,
)
from PrecipitationMunicipalAudit import (
    AGGREGATION_VERSION_ESPERADA,
    AUDIT_VERSION,
    auditar_precipitacion_municipal,
)

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str

    def display(valor):
        print(valor)

if AUDIT_VERSION != 'auditoria_precipitacion_municipal_v1':
    raise RuntimeError(f'Versión de auditoría inesperada: {AUDIT_VERSION}')

VARIABLE_NOMBRE = 'precipitacion'
DATASET_ID = 's54a-sgyg'
AGREGACION_ENTRADA = 'precipitacion_municipio_dia_2024_2025_v1'
AUDITORIA_NOMBRE = 'cierre_precipitacion_municipal_2024_2025_v1'
UMBRALES_LLUVIA_MM = [0.1, 1.0, 5.0, 10.0, 20.0]

EJECUTAR_AUDITORIA_MUNICIPAL = False
GUARDAR_RESULTADOS = True
SOBRESCRIBIR_AUDITORIA = False

PROCESSED_ROOT = (
    Path('/content/drive/MyDrive/eco2026_processed')
    if IN_COLAB
    else Path.cwd() / 'local_docs' / 'tmp'
)
INPUT_DIR = (
    PROCESSED_ROOT
    / 'clima_municipal'
    / f'variable={VARIABLE_NOMBRE}'
    / f'fuente={DATASET_ID}'
    / f'agregacion={slugificar(AGREGACION_ENTRADA)}'
)
OUTPUT_DIR = (
    PROCESSED_ROOT
    / 'auditorias_clima_municipal'
    / f'variable={VARIABLE_NOMBRE}'
    / f'fuente={DATASET_ID}'
    / f'auditoria={slugificar(AUDITORIA_NOMBRE)}'
)

print({
    'audit_version': AUDIT_VERSION,
    'ejecutar': EJECUTAR_AUDITORIA_MUNICIPAL,
    'guardar': GUARDAR_RESULTADOS,
    'entrada': str(INPUT_DIR),
    'salida': str(OUTPUT_DIR),
})

## 3. Plan y validación de entrada

In [ ]:
def leer_manifest_si_existe(ruta):
    return json.loads(ruta.read_text(encoding='utf-8')) if ruta.exists() else {}


def inspeccionar_plan_auditoria():
    manifest = leer_manifest_si_existe(INPUT_DIR / 'manifest.json')
    particiones = sorted(
        INPUT_DIR.glob(
            'departamento=*/anio=*/mes=*/precipitacion_municipio_dia.parquet'
        )
    )
    return pd.DataFrame([{
        'entrada_estado': manifest.get('estado', 'NO_ENCONTRADA'),
        'entrada_version': manifest.get('aggregation_version'),
        'particiones': len(particiones),
        'filas_manifest': manifest.get('metricas', {}).get('filas_municipio_dia'),
        'municipios_manifest': manifest.get('metricas', {}).get('municipios_objetivo'),
        'salida_existe': OUTPUT_DIR.exists(),
        'salida': str(OUTPUT_DIR),
    }])


plan_auditoria_df = inspeccionar_plan_auditoria()
display(Markdown('### Plan de auditoría municipal'))
display(plan_auditoria_df)

## 4. Funciones de carga, reporte, figuras y persistencia

In [ ]:
NOMBRES_SALIDA = {
    'cobertura_municipios': 'cobertura_municipios.parquet',
    'cobertura_periodos': 'cobertura_periodos.parquet',
    'cobertura_insuficiente': 'cobertura_insuficiente.parquet',
    'multiestacion_dias': 'multiestacion_dias.parquet',
    'resumen_multiestacion': 'resumen_multiestacion.parquet',
    'sensibilidad_anual': 'sensibilidad_media_mediana_anual.parquet',
    'sensibilidad_umbrales': 'sensibilidad_umbrales_lluvia.parquet',
    'grafica_cobertura': 'cobertura_temporal_municipios.html',
    'grafica_sensibilidad': 'sensibilidad_media_mediana.html',
    'reporte': 'AuditoriaMunicipal_precipitacion_2024_2025.md',
    'manifest': 'manifest.json',
}


def cargar_clima_municipal():
    manifest_path = INPUT_DIR / 'manifest.json'
    if not manifest_path.exists():
        raise FileNotFoundError(f'No existe el manifiesto municipal: {manifest_path}')
    manifest = leer_manifest_si_existe(manifest_path)
    if manifest.get('estado') != 'COMPLETA':
        raise RuntimeError('La agregación municipal no está COMPLETA.')
    if manifest.get('aggregation_version') != AGGREGATION_VERSION_ESPERADA:
        raise RuntimeError(
            f'Versión municipal inesperada: {manifest.get("aggregation_version")}'
        )
    archivos = sorted(
        INPUT_DIR.glob(
            'departamento=*/anio=*/mes=*/precipitacion_municipio_dia.parquet'
        )
    )
    if len(archivos) != 48:
        raise RuntimeError(f'Se esperaban 48 particiones y existen {len(archivos)}.')
    tabla = pd.concat([pd.read_parquet(archivo) for archivo in archivos], ignore_index=True)
    filas_manifest = manifest.get('metricas', {}).get('filas_municipio_dia')
    if filas_manifest is not None and len(tabla) != int(filas_manifest):
        raise RuntimeError(f'Filas leídas ({len(tabla):,}) != manifiesto ({filas_manifest:,}).')
    return tabla, manifest, archivos


def tabla_markdown(tabla, limite=None):
    vista = tabla.head(limite) if limite is not None else tabla
    try:
        return vista.to_markdown(index=False)
    except ImportError:
        return '```text\n' + vista.to_string(index=False) + '\n```'


def construir_figuras(resultado):
    try:
        import plotly.express as px
    except ImportError:
        print('Plotly no está disponible; se omiten las figuras.')
        return {}

    cobertura = resultado.cobertura_municipios.loc[
        resultado.cobertura_municipios['estaciones_canonicas_total'].gt(0)
    ].copy()
    figura_cobertura = px.bar(
        cobertura.sort_values('cobertura_sobre_dias_esperados_pct'),
        x='municipio',
        y='cobertura_sobre_dias_esperados_pct',
        color='departamento',
        hover_data=[
            'codigo_municipio', 'dias_validos',
            'dias_con_estacion_esperada', 'clasificacion_cobertura',
        ],
        title='Cobertura temporal de municipios con estación canónica',
        labels={'cobertura_sobre_dias_esperados_pct': 'Cobertura (%)'},
    )
    figura_cobertura.update_layout(height=560, xaxis_tickangle=-70)

    multi = resultado.multiestacion_dias
    figura_sensibilidad = px.scatter(
        multi,
        x='precipitacion_mediana_estaciones_mm',
        y='precipitacion_media_estaciones_mm',
        color='municipio',
        size='estaciones_con_dato',
        hover_data=['fecha', 'codigo_municipio', 'rango_estaciones_mm'],
        title='Sensibilidad diaria: mediana frente a media multiestación',
        labels={
            'precipitacion_mediana_estaciones_mm': 'Mediana (mm)',
            'precipitacion_media_estaciones_mm': 'Media (mm)',
        },
    )
    figura_sensibilidad.update_layout(height=620)
    return {
        'grafica_cobertura': figura_cobertura,
        'grafica_sensibilidad': figura_sensibilidad,
    }


def construir_reporte(resultado, manifest_entrada, inicio, fin, duracion):
    clasificacion = (
        resultado.cobertura_municipios['clasificacion_cobertura']
        .value_counts()
        .rename_axis('clasificacion_cobertura')
        .reset_index(name='municipios')
    )
    insuficiente = (
        resultado.cobertura_insuficiente
        .groupby(['departamento', 'codigo_municipio', 'municipio'], as_index=False)
        .agg(dias_cobertura_insuficiente=('fecha', 'size'))
        .sort_values('dias_cobertura_insuficiente', ascending=False)
    )
    return '\n'.join([
        '# Auditoría municipal de precipitación 2024-2025',
        '',
        f'- Contrato: `{AUDIT_VERSION}`',
        f'- Agregación auditada: `{manifest_entrada.get("aggregation_version")}`',
        f'- Commit ejecutor: `{detectar_commit(REPO_DIR)}`',
        f'- Commit de entrada: `{manifest_entrada.get("commit")}`',
        f'- Inicio: `{inicio.isoformat()}`',
        f'- Fin: `{fin.isoformat()}`',
        f'- Duración: `{formatear_duracion(duracion)}`',
        '',
        '> Auditoría de solo lectura. No elimina, imputa ni aprueba automáticamente la media, la mediana o un umbral de lluvia.',
        '',
        '## Métricas',
        '',
        tabla_markdown(pd.DataFrame([resultado.metricas])),
        '',
        '## Clasificación de cobertura municipal',
        '',
        tabla_markdown(clasificacion),
        '',
        '## Cobertura insuficiente por municipio',
        '',
        tabla_markdown(insuficiente),
        '',
        '## Sensibilidad de umbrales de lluvia',
        '',
        tabla_markdown(resultado.sensibilidad_umbrales_lluvia),
        '',
        '## Resumen de municipios multiestación',
        '',
        tabla_markdown(resultado.resumen_multiestacion),
        '',
        '## Mayor sensibilidad anual media-mediana',
        '',
        tabla_markdown(resultado.sensibilidad_media_mediana_anual, limite=25),
        '',
        '## Decisión',
        '',
        'La auditoría queda COMPLETA_CON_REVISION_PENDIENTE. Antes del paso 08 se deben revisar los municipios con mayor dispersión y definir cobertura mínima por periodo.',
        '',
    ])


def guardar_auditoria(resultado, manifest_entrada, archivos, reporte, figuras, inicio, fin, duracion):
    manifest_path = OUTPUT_DIR / NOMBRES_SALIDA['manifest']
    if manifest_path.exists() and not SOBRESCRIBIR_AUDITORIA:
        existente = leer_manifest_si_existe(manifest_path)
        if existente.get('estado') == 'COMPLETA_CON_REVISION_PENDIENTE':
            raise FileExistsError(f'La auditoría ya existe y no se sobrescribe: {OUTPUT_DIR}')

    tablas = {
        'cobertura_municipios': resultado.cobertura_municipios,
        'cobertura_periodos': resultado.cobertura_periodos,
        'cobertura_insuficiente': resultado.cobertura_insuficiente,
        'multiestacion_dias': resultado.multiestacion_dias,
        'resumen_multiestacion': resultado.resumen_multiestacion,
        'sensibilidad_anual': resultado.sensibilidad_media_mediana_anual,
        'sensibilidad_umbrales': resultado.sensibilidad_umbrales_lluvia,
    }
    salidas = {}
    for clave, tabla in tablas.items():
        ruta = OUTPUT_DIR / NOMBRES_SALIDA[clave]
        escribir_parquet_atomico(tabla, ruta, sobrescribir=SOBRESCRIBIR_AUDITORIA)
        salidas[clave] = {'ruta': str(ruta), 'filas': len(tabla)}

    reporte_path = OUTPUT_DIR / NOMBRES_SALIDA['reporte']
    escribir_texto_atomico(reporte, reporte_path, sobrescribir=SOBRESCRIBIR_AUDITORIA)
    for clave, figura in figuras.items():
        ruta = OUTPUT_DIR / NOMBRES_SALIDA[clave]
        escribir_texto_atomico(
            figura.to_html(full_html=True, include_plotlyjs='cdn'),
            ruta,
            sobrescribir=SOBRESCRIBIR_AUDITORIA,
        )
        salidas[clave] = {'ruta': str(ruta)}

    manifest = {
        'audit_version': AUDIT_VERSION,
        'estado': resultado.metricas['estado'],
        'commit': detectar_commit(REPO_DIR),
        'inicio': inicio.isoformat(),
        'fin': fin.isoformat(),
        'duracion_segundos': round(duracion, 2),
        'parametros': {'umbrales_lluvia_mm': UMBRALES_LLUVIA_MM},
        'entrada': {
            'ruta': str(INPUT_DIR),
            'commit': manifest_entrada.get('commit'),
            'aggregation_version': manifest_entrada.get('aggregation_version'),
            'particiones': len(archivos),
        },
        'metricas': resultado.metricas,
        'salidas': salidas,
        'reporte': str(reporte_path),
    }
    escribir_json_atomico(manifest, manifest_path, sobrescribir=True)
    print(f'Auditoría guardada en: {OUTPUT_DIR}')
    return manifest

## 5. Ejecución protegida

Con la bandera en `False` solo se muestra el plan. Con `True`, la auditoría lee las 48 particiones, genera evidencia y opcionalmente la guarda.

In [ ]:
resultado_auditoria_municipal = None

if not EJECUTAR_AUDITORIA_MUNICIPAL:
    print('Auditoría municipal desactivada. Revise el plan y active la bandera.')
else:
    inicio = ahora_proyecto()
    reloj = time.perf_counter()
    clima_municipal, manifest_entrada, archivos = cargar_clima_municipal()
    resultado_auditoria_municipal = auditar_precipitacion_municipal(
        clima_municipal,
        umbrales_lluvia_mm=UMBRALES_LLUVIA_MM,
    )
    figuras = construir_figuras(resultado_auditoria_municipal)
    fin = ahora_proyecto()
    duracion = time.perf_counter() - reloj
    reporte = construir_reporte(
        resultado_auditoria_municipal,
        manifest_entrada,
        inicio,
        fin,
        duracion,
    )

    display(Markdown('## Métricas de auditoría municipal'))
    display(pd.DataFrame([resultado_auditoria_municipal.metricas]))
    display(Markdown('## Cobertura por municipio'))
    display(resultado_auditoria_municipal.cobertura_municipios)
    display(Markdown('## Días con cobertura insuficiente'))
    display(resultado_auditoria_municipal.cobertura_insuficiente)
    display(Markdown('## Sensibilidad de umbrales de lluvia'))
    display(resultado_auditoria_municipal.sensibilidad_umbrales_lluvia)
    display(Markdown('## Resumen multiestación'))
    display(resultado_auditoria_municipal.resumen_multiestacion)
    for figura in figuras.values():
        figura.show()
    print(f'Duración: {formatear_duracion(duracion)}')

    if GUARDAR_RESULTADOS:
        guardar_auditoria(
            resultado_auditoria_municipal,
            manifest_entrada,
            archivos,
            reporte,
            figuras,
            inicio,
            fin,
            duracion,
        )
    else:
        print('Resultados no guardados porque GUARDAR_RESULTADOS=False.')

## 6. Cómo interpretar el resultado

- `COMPLETA_CON_REVISION_PENDIENTE` significa que la auditoría terminó, no que la regla municipal fue aprobada.
- Los acumulados de media y mediana se comparan usando exactamente los mismos días válidos; no extrapolan ausencias.
- Una diferencia grande puede representar heterogeneidad espacial real, estaciones no comparables o un problema de medición.
- Los municipios sin cobertura suficiente permanecen visibles y con `NaN`.
- El paso 08 solo se habilita después de documentar la decisión sobre dispersión y cobertura por periodo.